# A2A Task Delegation with LangGraph + Vertex AI

This notebook implements true A2A delegation:
**Controller -> Planner Agent -> Specialist Agent (Finance/Math)**.

## Objective
Build a planner agent that decides and delegates tasks to independent specialist agents discovered at runtime via `/agent-card`.

## Architecture
User -> Planner Agent (LLM) -> Finance Agent OR Math Agent -> Result

In [ ]:
import requests
import json
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph
from langchain_google_vertexai import ChatVertexAI

In [ ]:
# -----------------------------
# Vertex AI LLM
# -----------------------------
llm = ChatVertexAI(
    model="gemini-1.5-pro",
    temperature=0
)

In [ ]:
# -----------------------------
# State Definition
# -----------------------------
class State(TypedDict, total=False):
    user_input: str
    agents: List[Dict[str, Any]]
    plan: Dict[str, Any]
    result: Dict[str, Any]

In [ ]:
# -----------------------------
# Agent URLs
# -----------------------------
AGENT_URLS = [
    "http://localhost:8001",
    "http://localhost:8002"
]

In [ ]:
# -----------------------------
# Discovery
# -----------------------------
def discover_agents():
    agents = []

    for url in AGENT_URLS:
        try:
            res = requests.get(f"{url}/agent-card").json()
            agents.append(res)
        except Exception as e:
            print(f"[ERROR] {url}: {e}")

    return agents

In [ ]:
# -----------------------------
# Planner (Task Delegation)
# -----------------------------
def plan_and_delegate(user_input, agents):

    agent_descriptions = "\n".join([
        f"{a['name']}: {a['description']} | skills: {[s['name'] for s in a['skills']]}"
        for a in agents
    ])

    prompt = f"""
You are a PLANNER AGENT.

Your job:
1. Understand user request
2. Choose best agent
3. Return JSON

User request:
{user_input}

Available agents:
{agent_descriptions}

Return ONLY JSON:
{{
  "agent": "<agent_name>",
  "reason": "<why>"
}}
"""

    response = llm.invoke(prompt)

    return response.content

In [ ]:
# -----------------------------
# Nodes
# -----------------------------
def discovery_node(state: State):
    print("\n--- DISCOVERY ---")
    print("INPUT:", state)

    state["agents"] = discover_agents()

    print("OUTPUT:", state)
    return state


def planner_node(state: State):
    print("\n--- PLANNER ---")
    print("INPUT:", state)

    raw_plan = plan_and_delegate(
        state["user_input"],
        state["agents"]
    )

    print("RAW PLAN:", raw_plan)

    try:
        state["plan"] = json.loads(raw_plan)
    except:
        state["plan"] = {"agent": "finance-agent", "reason": "fallback"}

    print("OUTPUT:", state)
    return state


def delegation_node(state: State):
    print("\n--- DELEGATION ---")
    print("INPUT:", state)

    selected = state["plan"]["agent"]

    agent = next(
        a for a in state["agents"]
        if a["name"] == selected
    )

    res = requests.post(
        agent["endpoint"],
        json={"input": state["user_input"]}
    )

    state["result"] = res.json()

    print("OUTPUT:", state)
    return state

In [ ]:
# -----------------------------
# Graph
# -----------------------------
graph = StateGraph(State)

graph.add_node("discover", discovery_node)
graph.add_node("planner", planner_node)
graph.add_node("delegate", delegation_node)

graph.set_entry_point("discover")

graph.add_edge("discover", "planner")
graph.add_edge("planner", "delegate")

app = graph.compile()

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# -----------------------------
# Run
# -----------------------------
result = app.invoke({
    "user_input": "calculate interest for 1000"
})

print("\nFINAL RESULT:\n", result)